In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import cv2
import shutil
import pandas as pd
import numpy as np
import scipy.io as sio
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
MAT_FILE_PATH = "/content/drive/MyDrive/exam_v2/exams.mat"
SOURCE_SHEETS_DIR = "/content/drive/MyDrive/exam_v2/OMR_Dataset_Sheets"
YOLO_DIR = "YOLO_Dataset/"

# Thư mục nguồn của bạn hiện tại đã là chữ thường
SPLITS = ['train', 'val', 'test']

# Khởi tạo cấu trúc thư mục đích cho YOLO
for split in SPLITS:
    os.makedirs(os.path.join(YOLO_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR, 'labels', split), exist_ok=True)

# ==========================================
# 2. ĐỌC METADATA ĐỂ LẤY TỌA ĐỘ
# ==========================================
print("Đang tải dữ liệu từ exams.mat...")
mat_data = sio.loadmat(MAT_FILE_PATH, squeeze_me=True, struct_as_record=False)
struct_keys = [k for k in mat_data.keys() if not k.startswith('__')]
exam_structs = mat_data[struct_keys[0]]

# Tạo Dictionary tra cứu nhanh: {tên_ảnh: mảng_tọa_độ}
boxes_dict = {}
for exam in exam_structs:
    img_name = str(exam.imageName)
    if not img_name.endswith(('.png', '.jpg')):
        img_name += '.png'

    # Reshape mảng phẳng thành Nx4 [x, y, w, h] (Bản vá từ dữ liệu của bạn)
    rects = np.array(exam.questionRect).reshape(-1, 4)
    boxes_dict[img_name] = rects

# ==========================================
# 3. CHUYỂN ĐỔI SANG YOLO FORMAT (NORMALIZED)
# ==========================================
print("\nBắt đầu tạo Dataset YOLO...")

# Code giờ chỉ cần lặp trực tiếp qua danh sách 'train', 'val', 'test'
for split in SPLITS:
    split_path = os.path.join(SOURCE_SHEETS_DIR, split)
    if not os.path.exists(split_path):
        print(f"Lỗi: Không tìm thấy thư mục nguồn {split_path}")
        continue

    # Duyệt qua các thư mục exam0 -> exam5 bên trong train/val/test
    for exam_folder in os.listdir(split_path):
        exam_path = os.path.join(split_path, exam_folder)
        if not os.path.isdir(exam_path):
            continue

        # Duyệt qua từng ảnh phiếu
        for img_name in tqdm(os.listdir(exam_path), desc=f"Đang xử lý {split}/{exam_folder}"):
            src_img_file = os.path.join(exam_path, img_name)

            # Đọc ảnh để lấy Width và Height thực tế phục vụ chuẩn hóa
            img = cv2.imread(src_img_file)
            if img is None:
                continue
            img_h, img_w = img.shape[:2]

            # 3.1. Copy ảnh sang thư mục YOLO/images/
            dst_img_file = os.path.join(YOLO_DIR, 'images', split, img_name)
            shutil.copy2(src_img_file, dst_img_file)

            # 3.2. Tính toán và ghi file label .txt
            label_name = img_name.rsplit('.', 1)[0] + '.txt'
            label_file = os.path.join(YOLO_DIR, 'labels', split, label_name)

            boxes = boxes_dict.get(img_name, [])

            with open(label_file, 'w') as f:
                for box in boxes:
                    x, y, w, h = box

                    # Tính tọa độ tâm chuẩn hóa (Normalized Center Coordinates)
                    x_center = (x + w / 2.0) / img_w
                    y_center = (y + h / 2.0) / img_h
                    norm_w = w / img_w
                    norm_h = h / img_h

                    # Giới hạn dải an toàn (0.0 đến 1.0) để YOLO không báo lỗi Out of Bounds
                    x_center = max(0.0, min(1.0, x_center))
                    y_center = max(0.0, min(1.0, y_center))
                    norm_w = max(0.0, min(1.0, norm_w))
                    norm_h = max(0.0, min(1.0, norm_h))

                    # Ghi vào file với Class = 0 (answer_box)
                    f.write(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

print("\nHOÀN TẤT! Dữ liệu YOLO đã sẵn sàng.")

Đang tải dữ liệu từ exams.mat...

Bắt đầu tạo Dataset YOLO...


Đang xử lý test/exam5: 100%|██████████| 37/37 [00:29<00:00,  1.27it/s]


HOÀN TẤT! Dữ liệu YOLO đã sẵn sàng.


In [3]:
yaml_content = """
# data.yaml
path: ./YOLO_Dataset  # Đường dẫn gốc (có thể dùng đường dẫn tuyệt đối /content/YOLO_Dataset nếu chạy Colab)
train: images/train
val: images/val
test: images/test

# Số lượng class
nc: 1

# Tên của class
names: ['answer_box']
"""

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print("Created data.yaml successfully!")

Created data.yaml successfully!


In [4]:
!zip yolo_dataset.zip -r YOLO_Dataset

  adding: YOLO_Dataset/ (stored 0%)
  adding: YOLO_Dataset/images/ (stored 0%)
  adding: YOLO_Dataset/images/train/ (stored 0%)
  adding: YOLO_Dataset/images/train/exam5_351_1.png (deflated 2%)
  adding: YOLO_Dataset/images/train/exam5_161_1.png (deflated 2%)
  adding: YOLO_Dataset/images/train/exam5_1_1.png (deflated 2%)
  adding: YOLO_Dataset/images/train/exam1_56_1.png (deflated 4%)
  adding: YOLO_Dataset/images/train/exam1_76_1.png (deflated 4%)
  adding: YOLO_Dataset/images/train/exam1_51_1.png (deflated 4%)
  adding: YOLO_Dataset/images/train/exam3_10_1.png (deflated 5%)
  adding: YOLO_Dataset/images/train/exam0_21_1.png (deflated 5%)
  adding: YOLO_Dataset/images/train/exam1_62_1.png (deflated 4%)
  adding: YOLO_Dataset/images/train/exam5_98_1.png (deflated 3%)
  adding: YOLO_Dataset/images/train/exam5_209_1.png (deflated 2%)
  adding: YOLO_Dataset/images/train/exam5_144_1.png (deflated 3%)
  adding: YOLO_Dataset/images/train/exam2_20_1.png (deflated 4%)
  adding: YOLO_Dataset/i